# Gauthier_Cerema : Test Steger & Frangi

Ce notebook implémente les premiers tests demandés :
1. Téléchargement des données via `gdown` (pas de montage Drive).
2. Filtre Frangi classique (skimage) avec $\sigma=30$.
3. Calcul du Hessien Gaussien (implémentation GPU via PyTorch) avec $\sigma=30$ sur l'image spécifique 'Ombrage 05m base.tif'.
4. Affichage de la plus grande valeur propre ($\lambda_2$) en valeur absolue.

In [ ]:
!pip install rasterio torch torchvision gdown

# Téléchargement du dossier Gauthier_Cerema via gdown
# ID du dossier extrait du lien : 1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ
!gdown --folder https://drive.google.com/drive/folders/1lWi-7oZ064QSh_NjX_xrA5iv03A2hYjQ -O Gauthier_Cerema_Data

In [ ]:
import os
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from skimage.filters import frangi
import torch
import torch.nn.functional as F
import math

# --- DEFINITION DU CHEMIN ---
# Les données sont maintenant en local dans le dossier téléchargé
folder_path = 'Gauthier_Cerema_Data'

if os.path.exists(folder_path):
    print(f"Dossier trouvé : {folder_path}")
    files = os.listdir(folder_path)
    print("Fichiers :", files)
else:
    print(f"ATTENTION : Le dossier {folder_path} n'a pas été trouvé. Vérifiez le téléchargement gdown.")

In [ ]:
# --- IMPLEMENTATION STEGER / HESSIEN GPU ---
class StegerHessian:
    def __init__(self, sigma, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.sigma = sigma
        self.device = device
        self.kernels = self._precompute_kernels()

    def _precompute_kernels(self):
        # Determine kernel size (4*sigma)
        size = int(math.ceil(4 * self.sigma)) * 2 + 1
        x = torch.arange(-size // 2 + 1, size // 2 + 1, dtype=torch.float32, device=self.device)
        
        variance = self.sigma ** 2
        g_x = (1 / (math.sqrt(2 * math.pi) * self.sigma)) * torch.exp(-x**2 / (2 * variance))
        g_x_1 = -(x / variance) * g_x
        g_x_2 = ((x**2 / (variance**2)) - (1 / variance)) * g_x

        return {'0': g_x, '1': g_x_1, '2': g_x_2}

    def compute_hessian(self, image_tensor):
        if image_tensor.device != self.device:
            image_tensor = image_tensor.to(self.device)

        k0 = self.kernels['0'].view(1, 1, 1, -1)
        k1 = self.kernels['1'].view(1, 1, 1, -1)
        k2 = self.kernels['2'].view(1, 1, 1, -1)

        k0_T = k0.transpose(2, 3)
        k1_T = k1.transpose(2, 3)
        k2_T = k2.transpose(2, 3)

        pad = k0.shape[3] // 2
        
        # Ixx
        ixx_temp = F.conv2d(image_tensor, k2, padding=(0, pad))
        ixx = F.conv2d(ixx_temp, k0_T, padding=(pad, 0))

        # Iyy
        iyy_temp = F.conv2d(image_tensor, k0, padding=(0, pad))
        iyy = F.conv2d(iyy_temp, k2_T, padding=(pad, 0))

        # Ixy
        ixy_temp = F.conv2d(image_tensor, k1, padding=(0, pad))
        ixy = F.conv2d(ixy_temp, k1_T, padding=(pad, 0))
        
        return ixx, ixy, iyy

    def compute_eigenvalues(self, ixx, ixy, iyy):
        # Trace & Determinant
        trace = ixx + iyy
        det = ixx * iyy - ixy**2
        disc = torch.sqrt((ixx - iyy)**2 + 4 * ixy**2)
        
        l1 = (trace - disc) / 2
        l2 = (trace + disc) / 2
        return l1, l2

In [ ]:
# --- 1. FILTRE FRANGI CLASSIQUE (tous les tifs) ---
sigma_val = 30

tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif')]

for file_name in tif_files:
    file_path = os.path.join(folder_path, file_name)
    try:
        with rasterio.open(file_path) as src:
            data = src.read(1)
            
            # Frangi Classique
            print(f"Traitement Frangi (sigma={sigma_val}) sur {file_name}...")
            response = frangi(data, sigmas=[sigma_val])

            plt.figure(figsize=(10, 6))
            plt.imshow(response, cmap='gray')
            plt.colorbar(label='Frangi Response')
            plt.title(f"Frangi (sigma={sigma_val}) : {file_name}")
            plt.show()
    except Exception as e:
        print(f"Erreur lecture {file_name}: {e}")

In [ ]:
# --- 2. HESSIEN GAUSSIEN (Cible : Ombrage 05m base.tif) ---
target_file = "Ombrage 05m base.tif"
target_path = os.path.join(folder_path, target_file)

if os.path.exists(target_path):
    print(f"Calcul du Hessien Gaussien sur {target_file}...")
    
    with rasterio.open(target_path) as src:
        data = src.read(1)
        # Normalisation
        data = data.astype(np.float32)
        if data.max() > 0:
            data = (data - data.min()) / (data.max() - data.min())
            
        # Conversion PyTorch
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        img_tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(device)
        
        # Calcul Hessien
        steger = StegerHessian(sigma=sigma_val, device=device)
        ixx, ixy, iyy = steger.compute_hessian(img_tensor)
        l1, l2 = steger.compute_eigenvalues(ixx, ixy, iyy)
        
        # Plus grande valeur propre en valeur absolue
        abs_l1 = torch.abs(l1)
        abs_l2 = torch.abs(l2)
        max_eigen = torch.maximum(abs_l1, abs_l2)
        
        result_np = max_eigen.squeeze().cpu().numpy()
        
        plt.figure(figsize=(12, 10))
        plt.imshow(result_np, cmap='inferno')
        plt.colorbar(label='Max Absolute Eigenvalue')
        plt.title(f"Hessian Eigenvalue Response (sigma={sigma_val}) : {target_file}")
        plt.show()
else:
    print(f"Fichier cible {target_file} introuvable dans {folder_path}")